In [1]:
from models.parameters import Parameter
from typing import List, Tuple, Union, Callable, Dict

def test(a,b=2):
    return a + b

#### NUOVA CLASSE PER IL PARAMETER HANDLER

In [22]:
from types import UnionType
from typing import Iterator
from collections import OrderedDict
import numpy as np

# Assuming 'Parameter' is defined elsewhere
# from parameter_module import Parameter


class ParameterHandler:
    """
    Classe che gestisce un insieme di parametri per un modello.

    Attributes:
        _parameters (OrderedDict[str, Parameter]): Dizionario dei parametri.
        _is_inside_model (bool): Indica se il gestore è stato aggiunto a un modello.
        _cache (Dict[str, Any]): Cache per le proprietà dei parametri.
    """

    def __init__(
        self, parameters: Union["Parameter", List["Parameter"]] = None
    ) -> None:
        """
        Inizializza il gestore dei parametri.

        Args:
            parameters (Union[Parameter, List[Parameter]], opzionale): Un singolo parametro o una lista di parametri.
        """
        self._parameters = OrderedDict()
        self._is_inside_model = (
            False  # Una volta dentro modello non posso aggiungere parametri
        )
        self._cache = {}

        

        if isinstance(parameters, Parameter):
            self.add_parameter(parameters)
        elif isinstance(parameters, list):
            for param in parameters:
                self.add_parameter(param)
        elif parameters is None:
            pass
        else:
            raise TypeError(
                "Parameters must be of type Parameter or List[Parameter]",
                type(parameters),
            )

    def _invalidate_cache(self) -> None:
        """
        Invalida la cache dei parametri.
        """
        self._cache = {}

    def _update_cache(self, key, value) -> None:
        """aggiorna la cache dei parametri.
        
        PARAMETRI CACHATI:
        1- self.parameter_names
        2- self.parameter_values
        3- self.parameters_bounds
        4- self.n_free_parameters
        
        """
        if key not in [
            "parameters_names",
            "parameters_values",
            "parameters_bounds",
            "parameters_keys",
            "parameters_values_dict",
        ]:
            raise ValueError('Cache error')
        self._cache[key] = value
        

    @property
    def is_inside_model(self) -> bool:
        """
        Indica se il gestore è stato aggiunto a un modello.

        Returns:
            bool: True se è stato aggiunto a un modello, False altrimenti.
        """
        return self._is_inside_model

    def lock(self) -> None:
        """
        Blocca il gestore per prevenire l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = True

    def unlock(self) -> None:
        """
        Sblocca il gestore per permettere l'aggiunta di nuovi parametri.
        """
        self._is_inside_model = False

    @property
    def parameters_values(self) -> List[float]:
        """
        Ritorna i valori dei parametri.

        Returns:
            List[float]: Lista dei valori dei parametri.
        """
        if 'parameters_values' in self._cache:
            return self._cache['parameters_values']
        else:
            return [p.value for p in self]

    @property
    def parameters_names(self) -> List[str]:
        """
        Ritorna i nomi dei parametri.

        Returns:
            List[str]: Lista dei nomi dei parametri.
        """
        if "parameters_names" in self._cache:
            return self._cache["parameters_names"]
        else:
            return [p.name for p in self]
    
    @property
    def parameters_keys(self) -> List[str]:
        """
        Ritorna i nomi dei parametri, dentro il dizionario.

        Returns:
            List[str]: Lista dei nomi dei parametri.
        """
        if "parameters_keys" in self._cache:
            return self._cache["parameters_keys"]
        else:
            return [p for p in self._parameters.keys()]
        
    @property
    def parameters_values_dict(self):
        """
        Ritorna il dizionario key-value dei parametri, dove key
        non è necessariamente il nome del parametro
        __low level, meglio non giocarci.

        Returns:
            List[str]: Lista dei nomi dei parametri.
        """
        if "parameters_values_dict" in self._cache:
            return self._cache["parameters_values_dict"]
        return {
            key: val for key, val in zip(self.parameters_keys, self.parameters_values)
        }

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:
        """
        Ritorna i limiti dei parametri.

        Returns:
            List[Tuple[float, float]]: Lista dei limiti dei parametri.
        """
        
        return [p.bounds for p in self]

    @property
    def n_free_params(self) -> int:
        """
        Ritorna il numero di parametri liberi.

        Returns:
            int: Numero di parametri liberi.
        """
        return len(self.free_parameters)

    @property
    def free_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri liberi.

        Returns:
            List[Parameter]: Lista dei parametri liberi.
        """
        if "free_parameters" not in self._cache:
            self._cache["free_parameters"] = [p for p in self if not p.frozen]
        return self._cache["free_parameters"]

    @property
    def frozen_parameters(self) -> List["Parameter"]:
        """
        Ritorna solo i parametri congelati.

        Returns:
            List[Parameter]: Lista dei parametri congelati.
        """
        if "frozen_parameters" not in self._cache:
            self._cache["frozen_parameters"] = [p for p in self if p.frozen]
        return self._cache["frozen_parameters"]
    
    def _map_name_to_index(self, key: str) -> int:
        """
        Mappa il nome di un parametro al corrispondente indice.

        Args:
            key (str): Nome del parametro.

        Returns:
            int: Indice del parametro.

        Raises:
            KeyError: Se il nome del parametro non è trovato.
        """
        try:
            return self.parameters_keys.index(key)
        except ValueError:
            raise KeyError(f"Key '{key}' not found in the dictionary")

    def _map_indices_to_names(self, index: int) -> str:
        """
        Mappa l'indice di un parametro al corrispondente nome.

        Args:
            index (int): Indice del parametro.

        Returns:
            str: Nome del parametro.

        Raises:
            IndexError: Se l'indice è fuori dai limiti.
        """
        
        if index < 0 or index >= len(self.parameters_keys):
            raise IndexError(f"Index '{index}' is out of bounds for the dictionary")
        return self.parameters_keys[index]

    def set_values(
        self, values: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i valori dei parametri.

        Args:
            values (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(values, "value", include_frozen=include_frozen)
        self._update_cache(key = 'parameters_values', value =[p.value for p in self])
        self._update_cache(
            key="parameters_values_dict",
            value={
                key: val
                for key, val in zip(self.parameters_keys, self.parameters_values)
            },
        )
        

    def set_bounds(
        self, bounds: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta i limiti dei parametri.

        Args:
            bounds (Union[List, Dict]): Limiti da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.
        """
        self._assign_attribute(bounds, "bounds", include_frozen=include_frozen)
        self._update_cache("parameters_bounds", [p.bounds for p in self])

    def set_frozen(
        self, is_frozen: Union[List, Dict], include_frozen: bool = False
    ) -> None:
        """
        Imposta lo stato di congelamento dei parametri.

        Args:
            is_frozen (Union[List, Dict]): Stato di congelamento da assegnare (lista o dizionario).
            include_frozen (bool, opzionale): Se includere i parametri già congelati. Default è False.
        """
        self._assign_attribute(is_frozen, "frozen", include_frozen=include_frozen)
        #self._update_cache("parameters_values", [p.value for p in self])

    def _assign_attribute(
        self, items: Union[List, Dict], attribute: str, include_frozen: bool = False
    ) -> None:
        """
        Assegna un valore a un attributo dei parametri.

        Args:
            items (Union[List, Dict]): Valori da assegnare (lista o dizionario).
            attribute (str): Nome dell'attributo da assegnare.
            include_frozen (bool, opzionale): Se includere i parametri congelati. Default è False.

        Raises:
            ValueError: Se il numero di elementi non corrisponde al numero di parametri.
            TypeError: Se items non è né una lista né un dizionario.
        """
        if isinstance(items, (list, np.ndarray)):
            params = list(self) if include_frozen else self.free_parameters
            if len(items) != len(params):
                raise ValueError(
                    f"Number of items {len(items)} must match number of {'all' if include_frozen else 'free'} parameters ({len(params)})"
                )
            for param, val in zip(params, items):
                setattr(param, attribute, val)
        elif isinstance(items, dict):
            for name, val in items.items():
                param = self[name]
                if not include_frozen and param.frozen:
                    continue
                setattr(param, attribute, val)
        else:
            raise TypeError("Items must be a list or dictionary")

    def add_parameter(self, parameter: "Parameter", name = None) -> None:
        """
        Aggiunge un parametro al gestore.

        Args:
            parameter (Parameter): Il parametro da aggiungere.

        Raises:
            ValueError: Se il parametro esiste già o se si tenta di aggiungere un parametro dopo la creazione del modello.
        """
        if name is None:
            name = parameter.name

        if self._is_inside_model:
            raise ValueError(
                f"Cannot add parameter {name} to model after it is locked"
            )
        if not isinstance(parameter, Parameter):
            raise TypeError('Addes parameter must be istance of Parameter class')
        
        if name in self._parameters:
            raise ValueError(f"Parameter {name} already exists.")
        
        #if name is None:
        #    name = parameter.name

        self._parameters[name] = parameter
        #self.parameter_map[parameter.name] = parameter.name
        
        #self._invalidate_cache()

    def __getitem__(self, name: str) -> "Parameter":
        """
        Ritorna un parametro usando il suo nome.

        Args:
            name (str): Nome del parametro.

        Returns:
            Parameter: Il parametro richiesto.

        Raises:
            KeyError: Se il parametro non è trovato.
        """
        try:
            return self._parameters[name]
        except KeyError:
            raise KeyError(f"Parameter '{name}' not found.")

    def __setitem__(self, key: str, value: "Parameter") -> None:
        """
        Imposta un parametro usando il suo nome.

        Args:
            key (str): Nome del parametro.
            value (Parameter): Il parametro da impostare.

        Raises:
            ValueError: Se si tenta di impostare un parametro dopo la creazione del modello.
            TypeError: Se value non è un'istanza di Parameter.
            ValueError: Se il parametro esiste già.
        """
        #if self._is_inside_model:
        #    raise ValueError(f"Cannot add parameter {key} to model after it is locked")
        if not isinstance(value, Parameter):
            raise TypeError(
                f"New parameter must be instance of Parameter, not {type(value)}"
            )
        if key in self._parameters:
            raise ValueError(f"Parameter {key} already exists.")
        self._parameters[key] = value
        self._invalidate_cache()

    def __contains__(self, key: str) -> bool:
        """
        Verifica se un parametro è presente usando il suo nome.

        Args:
            key (str): Nome del parametro.

        Returns:
            bool: True se il parametro è presente, False altrimenti.
        """
        return key in self._parameters

    def __iter__(self) -> Iterator["Parameter"]:
        """
        Itera sui parametri.

        Returns:
            Iterator[Parameter]: Iteratore sui parametri.
        """
        return iter(self._parameters.values())

    def __len__(self) -> int:
        """
        Ritorna il numero di parametri.

        Returns:
            int: Numero di parametri.
        """
        return len(self._parameters)
    
    def __str__(self) -> str:
        """
        Ritorna una rappresentazione in formato tabella dei parametri gestiti.

        Returns:
            str: Tabella che mostra nome, valore, bounds e stato frozen dei parametri.
        """
        # Creiamo l'header della tabella
        header = f"{'Name':<20} {'Value':<20} {'Bounds':<30} {'Frozen':<10}\n"
        header += "-" * 60 + "\n"
        
        # Raccogliamo i dati di ogni parametro
        rows = []
        for i, param in enumerate(self):
            name = self.parameters_names[i]
            value = str(param.value) if param.value is not None else "None"
            bounds = str(param.bounds) if param.bounds is not None else "None"
            frozen = "Yes" if param.frozen else "No"
            
            # Aggiungiamo una riga alla tabella
            row = f"{name:<20} {value:<20} {bounds:<30} {frozen:<10}"
            rows.append(row)
        
        # Uniamo l'header con tutte le righe
        table = header + "\n".join(rows)
        return table

    def items(self):
        """
        Ritorna gli elementi del gestore come coppie chiave-valore.

        Returns:
            ItemsView: Vista degli elementi del gestore.
        """
        return self._parameters.items()

    def keys(self):
        """
        Ritorna le chiavi del dizionario base.

        Returns:
            KeysView: Vista delle chiavi del gestore.
        """
        return self._parameters.keys()

    def values(self):
        """
        Ritorna i parametri.

        Returns:
            ValuesView: Vista dei valori del gestore.
        """
        return self._parameters.values()
    
    
    
    
    
    


# Esempi di utilizzo
p1 = Parameter(name="param1", value=5.0, bounds=(0.0, 10.0), frozen=True)
p2 = Parameter(name="param2", value=7.0, bounds=(0.0, 10.0))
p3 = Parameter(name = 'param3', value =3.5)
handler = ParameterHandler([p1, p2])

#handler.add_parameter(p3, name = 'test')
print(handler.parameters_names)
print(handler._map_name_to_index('param1'))
print([p.name for p in handler])


['param1', 'param2']
0
['param1', 'param2']


TypeError: ('Value must be of type Number, not ', <class 'str'>)

In [3]:
print(handler.parameters_names)  # ['param1', 'param2']
print(handler.parameters_values)  # [5.0, 7.0]
print(handler.n_free_params)  # 1
print(
    handler.free_parameters
)  # [<Parameter name='param2', value=7.0, bounds=(0.0, 10.0), frozen=False>]
print(
    handler.frozen_parameters
)  # [<Parameter name='param1', value=5.0, bounds=(0.0, 10.0), frozen=True>]

handler.set_values({"param2": 8.0})
print(handler.parameters_values)  # [5.0, 8.0]

handler.set_bounds({"param2": (0.0, 15.0)})
print(handler.parameters_bounds)  # [(0.0, 10.0), (0.0, 15.0)]

handler.set_frozen({"param1": True})
print(handler.frozen_parameters)  # [<Parameter name='param1', val

print("param1" in handler, len(handler))

handler.set_frozen(is_frozen=[False, False], include_frozen=True)
handler["param1"].value = 2.55  # metodo migliore
print(handler.free_parameters)
print(handler["param1"])
print(handler["param2"])
print(handler)
print(handler.keys())
# ['parameters_names, parameters_values, parameters_bounds']:

['param1', 'param2']
[5.0, 7.0]
1
[5.0, 8.0]
[(0.0, 10.0), (0.0, 15.0)]
True 2
PARAM NAME: param1
------------------------------------------------------------
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
------------------------------------------------------------
param1          2.55       0          (0, 10)                                   

PARAM NAME: param2
------------------------------------------------------------
NOME            VALORE     FREEZE     BOUNDS               DESCRIZIONE          
------------------------------------------------------------
param2          8          0          (0, 15)                                   

Name                 Value                Bounds                         Frozen    
------------------------------------------------------------
param1               2.55                 (0.0, 10.0)                    No        
param2               8.0                  (0.0, 15.0)                    No        


In [4]:
class ParameterMapper:
    '''classe nodo per ParameterHandler, che segue le struttura di CompositeModel in modo da mappare i parametri'''
    def __init__(self, left, right) -> None:
        self._left = left
        self._right = right

        self.param_map = OrderedDict() # (nome, Parameter)
    
    @property
    def left(self):
        return self._left

    @property
    def right(self):
        return self._right
    

In [5]:
def gaussian(x, mu, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def calcola_dimensioni(lista):
    # Variabili di controllo per la sequenza
    dimensioni = 0
    if lista[0] == "x" and lista[1] != "y":
        dimensioni += 1
    elif lista[0] == "x" and lista[1] == "y" and lista[2] != "z":
        dimensioni += 2
    elif lista[0] == "x" and lista[1] == "y" and lista[2] == "z":
        dimensioni += 3

    return dimensioni

def componemodels(op, **kwargs):
    return lambda left, right: CompositeModel(left, right, op, **kwargs)

In [6]:
import inspect
from copy import deepcopy
import warnings

class Model:
    def __init__(
        self, func, parameters, ndim, ninputs, noutputs, name="SimpleModel"
    ) -> None:
        
        self._parameters = parameters
        self._ndim = ndim
        self._ninputs = ninputs
        self._n_outputs = noutputs
        self._callable = func
        self._name = name
        self._grid_variables = None  # Inizializza _grid_variables nel costruttore
        
        #self._update_callable()
        self._cache = {}    # cache base 
    
    def _update_cache(self, key, value) ->None:
        if key not in [
            "binary_freeze_map",
            "binary_melt_map",
            "frozen_indeces",
            "not_frozen_indeces",
            'parameters_values_dict',
        ]:
            raise ValueError('cache error')
        self._cache[key] = value

    # POINTER PROPRIETA
    @property
    def parameters(self):
        return self._parameters

    @property
    def name(self):
        return self._name
    
    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise TypeError('New name must be of type string')
        self._name = value

    @property
    def n_dim(self):
        return self._ndim

    @property
    def n_inputs(self):
        return self._ninputs

    @property
    def n_outputs(self):
        return self._n_outputs

    @property
    def grid_variables(self):
        return self._grid_variables
    
    @property
    def parameters_names(self) -> List[str]:    #cached
        return self.parameters.parameters_names

    @property
    def parameters_keys(self) -> List[str]:
        return self.parameters.parameters_keys
    @property
    def n_parameters(self) -> int:
        return len(self.parameters)

    @property
    def parameters_values(self) -> List[float]: #cached
        return self.parameters.parameters_values

    @property
    def parameters_bounds(self) -> List[Tuple[float, float]]:        
        return self.parameters.parameters_bounds

    @property
    def free_parameters(self) -> List[Parameter]:       
        return self.parameters.free_parameters
    
    @property
    def parameters_values_dict(self):
        return self.parameters.parameters_values_dict

    @property
    def frozen_parameters(self) -> List[Parameter]:
        return self.parameters.frozen_parameters

    @property
    def n_free_parameters(self) -> int:
        return self.parameters.n_free_params

    @property
    def _binary_freeze_map(self) -> List[bool]:
        # possibile da cachare
        if 'binary_freeze_map' in self._cache:
            return self._cache["binary_freeze_map"]
        return [p.frozen for p in self]

    @property
    def _binary_melt_map(self) -> List[bool]:
        # possibile da cachare
        if "binary_melt_map" in self._cache:
            return self._cache["binary_melt_map"]
        return [not p.frozen for p in self]
    
    @property
    def left(self):
        return None
    
    @property
    def right(self):
        return None
    
    @property
    def not_frozen_indeces(self):
        if "not_frozen_indeces" in self._cache:
            return self._cache["not_frozen_indeces"]
        return [
            i
            for i in range(len(self._binary_freeze_map))
            if self._binary_freeze_map[i] is False
        ]

    @property
    def frozen_indeces(self):
        if "frozen_indeces" in self._cache:
            return self._cache["frozen_indeces"]
        return [
            i 
            for i in range(len(self._binary_freeze_map)) 
            if self._binary_freeze_map[i] is True
        ]
    
    def _map_name_to_index(self,name) -> int:
        return self.parameters._map_name_to_index(name)
    
    def set_parameters_values(self, args=None, **kwargs) -> None:
        """
        Imposta i valori dei parametri utilizzando argomenti posizionali o parole chiave.

        Args:
            args (list, opzionale): Una lista di valori per i parametri.
            kwargs (dict, opzionale): Un dizionario con nomi di parametri come chiavi e valori corrispondenti.

        

        Esempio:
            >>> obj.set_parameters_values([1, 2, 3])
            >>> obj.set_parameters_values(param1=1, param2=2)
        """
        if args:
            self.parameters.set_values(args)
        if kwargs:
            self.parameters.set_values(kwargs)

    
    def set_parameters_bounds(self, args=None, **kwargs) -> None:
        """
        Imposta i limiti dei parametri utilizzando argomenti posizionali o parole chiave.

        Args:
            args (list, opzionale): Una lista di limiti per i parametri.
            kwargs (dict, opzionale): Un dizionario con nomi di parametri come chiavi e limiti corrispondenti.


        Esempio:
            >>> obj.set_parameter_bounds([0, 10])
            >>> obj.set_parameter_bounds(param1=(0, 10), param2=(0, 5))
        """
        if args:
            self.parameters.set_bounds(args)
        if kwargs:
            self.parameters.set_bounds(kwargs)
        
    
    def _set_frozen_state(self, state: bool, *args) -> None:
        """
        Imposta lo stato di congelamento per i parametri specificati o per tutti i parametri.

        Args:
            state (bool): Stato di congelamento (True per congelare, False per scongelare).
            args (tuple): Una lista di nomi o indici dei parametri da congelare/scongelare.

        Esempio:
            >>> obj._set_frozen_state(True, 'param1', 'param2')
            >>> obj._set_frozen_state(False)
        """
        if not args:
            vals = self.parameters_keys
            
        else:
            vals = args
        for element in vals:
            name = element
            if isinstance(element, int):
                name = self.parameters._map_indices_to_names(element)
            self.parameters[name].frozen = state


    def freeze_parameters(self, *args, **kwargs) -> None:
        """
        Congela i parametri specificati o tutti i parametri se nessuno è specificato.

        Args:
            args (tuple): Una lista di nomi o indici dei parametri da congelare.
            kwargs (dict): Un dizionario con nomi di parametri come chiavi e valori corrispondenti per congelarli a determinati valori.

        Esempio:
            >>> obj.freeze_parameters('param1', 'param2')
            >>> obj.freeze_parameters(param1=1, param2=2)
        """
        if kwargs:
            # posso freezare un parametro ad un determinato valore
            self.set_parameters_values(kwargs)
            args = [*args, *list(kwargs.keys())]

        self._set_frozen_state(True, *args)

        # AGGIORNO CACHE
        self._update_cache(key = 'binary_freeze_map', value=[p.frozen for p in self])
        self._update_cache(key="binary_melt_map", value=[not p.frozen for p in self])
        self._update_cache(
            key="not_frozen_indeces",
            value=[
                i
                for i in range(len(self._binary_freeze_map))
                if self._binary_freeze_map[i] is False
            ],
        )
        self._update_cache(
            key="frozen_indeces",
            value=[
                i
                for i in range(len(self._binary_freeze_map))
                if self._binary_freeze_map[i] is True
            ],
        )

    def unfreeze_parameters(self, *args) -> None:
        """
        Scongela i parametri specificati o tutti i parametri se nessuno è specificato.

        Args:
            args (tuple): Una lista di nomi o indici dei parametri da scongelare.

        Esempio:
            >>> obj.unfreeze_parameters('param1', 'param2')
            >>> obj.unfreeze_parameters()
        """
        self._set_frozen_state(False, *args)

        # AGGIORNO CACHE
        self._update_cache(key="binary_freeze_map", value=[p.frozen for p in self])
        self._update_cache(key="binary_melt_map", value=[not p.frozen for p in self])
        self._update_cache(
            key="not_frozen_indeces",
            value=[
                i
                for i in range(len(self._binary_freeze_map))
                if self._binary_freeze_map[i] is False
            ],
        )
        self._update_cache(
            key="frozen_indeces",
            value=[
                i
                for i in range(len(self._binary_freeze_map))
                if self._binary_freeze_map[i] is True
            ],
        )



    @staticmethod
    def _extract_params(method, default_value = 1, **kwargs):
        

        """
        Estrae i nomi e i valori di default dei parametri dal metodo evaluate.

        Parameters:
        -----------
        method : function
            Metodo evaluate della classe.

        Returns:
        --------
        tuple
            Lista dei nomi dei parametri, dei valori di default e dello stato frozen.
        """
        signature = inspect.signature(method)
        params = {}
        is_constant = []
        for param_name, param in signature.parameters.items():
            if param_name != "self":
                if param.default is inspect.Parameter.empty:
                    params[param_name] = default_value
                    is_constant.append(False)
                else:
                    params[param_name] = param.default
                    is_constant.append(True)
        if kwargs:
            print(kwargs)
            for key,value in kwargs.items():
                if key not in params and key != 'name':
                    raise ValueError(f'Param {key} is not a function-key')
                
                params[key] = value
                
        return list(params.keys()), list(params.values()), is_constant

    @classmethod
    def from_callable(cls, func, ndim = None, ninputs = None,
                      noutputs = 1, default_value = 1, name="SimpleModel", **kwargs):
        names, values, frozen = cls._extract_params(func, default_value=default_value, **kwargs)

        # check sul numero di dimensioni
        if ndim is None:
            ndim = calcola_dimensioni(names)
        
        if ninputs is None:
            ninputs = len(names) - calcola_dimensioni(names)


        parameters = ParameterHandler()

        for i in range(ndim, len(names)):
            parameters.add_parameter(
                Parameter(name=names[i], value=values[i], frozen=frozen[i])
            )
        # func, parameters, ndim, ninputs, noutputs, name="SimpleModel"

        new_cls = cls(func, parameters, ndim, ninputs, noutputs, name)
        new_cls._grid_variables = names[:ndim]  # Sovrascrive _grid_variables
        if ndim == 0:
            new_cls._ndim = 1

        
        new_cls._ninputs = len(parameters)
        #new_cls._update_callable()
        return new_cls
    
    

    def __str__(self):
        """
        Restituisce una rappresentazione testuale del modello.

        Returns:
            str: Una stringa che rappresenta il modello.
        """
        total_string = f"MODEL NAME: {self.name} \n"
        total_string += f"FREE PARAMS: {self.n_free_parameters}\n"
        total_string += f"GRID VARIABLES: {self.grid_variables}\n"
        total_string +=  f"N-DIM: {self.n_dim}\n"
        total_string += "-" * 60 + "\n"
        total_string += (
            f"{'':<4} {'NAME':<15} {'VALUE':<10} {'IS-FROZEN':<10} {'BOUNDS':<20}\n"
        )
        total_string += "-" * 60 + "\n"
        for i, param in enumerate(self._parameters):
            value_str = f"{param.value:.2f}"
            bounds_str = f"({param.bounds[0]:.2f}, {param.bounds[1]:.2f})"
            frz = "Yes" if param.frozen else "No"
            total_string += (
                f"{i:<4} {param.name:<15} {value_str:<10} {frz:<10} {bounds_str:<20}\n"
            )
        return total_string
    
    
    def evaluate(self, *args, **kwargs):
        return self._callable(*args, **kwargs)
    
    def __call__(self, **kwargs):
        
        grid = []
        for key in self.grid_variables:
            grid.append(kwargs.pop(key))
            
        tmp = self.parameters_values_dict
        if kwargs:            
            for key in kwargs:
                if key not in self.grid_variables and self[key].frozen:
                    warnings.warn(f'Parameter {key} is frozen, new value will be ignored')
                    continue
                tmp[key] = kwargs[key]
                
        return self._callable(*grid,  **tmp)

        
             

    def __getitem__(self, name: str) -> Parameter:
        return self.parameters[name]

    def __setitem__(self, key, value: Parameter) -> None:
        return self.parameters.__setitem__(key, value)

    def __contains__(self, key: str) -> bool:
        return self.parameters.__contains__(key)

    def __iter__(self):
        return self.parameters.__iter__()

    def __len__(self) -> int:
        return self.parameters.__len__()
    
    def copy(self):
        return deepcopy(self)
    
    
    
    __add__ = componemodels("+")
    __mul__ = componemodels("*")
    __or__ = componemodels("|")
    __truediv__ = componemodels("/")
    __sub__ = componemodels("-")
    __pow__ = componemodels('**')

    

In [7]:
def simple_test(x,y,a,b,c):
    return a+b+c

mod = Model.from_callable(simple_test)
mod['c'].value = 10
mod['c'].frozen = True

print(mod._map_name_to_index('c'))
print(mod(x=0,y=0, a = 2.22)) 
print(mod.evaluate(1,1,1,1,1))
print(mod)

2
13.22
3
MODEL NAME: SimpleModel 
FREE PARAMS: 2
GRID VARIABLES: ['x', 'y']
N-DIM: 2
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    a               1.00       No         (-inf, inf)         
1    b               1.00       No         (-inf, inf)         
2    c               10.00      Yes        (-inf, inf)         



In [8]:
model1 = Model.from_callable(gaussian, name='Model1')
model1.freeze_parameters(1)
print(model1)
print(model1(x = 1))
print('--'*5)

model2 = Model.from_callable(gaussian, name = 'Model2', ndim = 1)
print(model2)
print(model2.evaluate(0,2,2))


MODEL NAME: Model1 
FREE PARAMS: 1
GRID VARIABLES: ['x']
N-DIM: 1
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    mu              1.00       No         (-inf, inf)         
1    sigma           1.00       Yes        (-inf, inf)         

0.3989422804014327
----------
MODEL NAME: Model2 
FREE PARAMS: 2
GRID VARIABLES: ['x']
N-DIM: 1
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS              
------------------------------------------------------------
0    mu              1.00       No         (-inf, inf)         
1    sigma           1.00       No         (-inf, inf)         

0.12098536225957168


In [9]:
import operator

class CompositeModel(Model):

    LINEAR_OPERATIONS = ["+", "-", "*", "/", "**"]
    COMPOSITE_OPERATION = "|"

    IS_COMPOSITE = True
    _parameters = OrderedDict()
    _name = 'CompositeModel'
    _callable = None

    def __init_subclass__(cls) -> None:
        return super().__init_subclass__()

    def __init__(self, left = None, right = None, op = '+') -> None:
        self._left = left
        self._right = right
        self.op_str = op
        self._operator = self.map_operator(op)
        
        self._n_dim, self._n_inputs, self._n_outputs = self._update_n_dim()
        self._parameters = self._init_parameters()
        self.submodels = self._collect_submodels()
        self._cache = {}
    
    @property
    def left(self):
        return self._left
    
    @property
    def right(self):
        return self._right
    
    @property
    def parameters(self):
        return self._parameters
    
    @property
    def n_dim(self):
        return self._n_dim

    @property
    def n_inputs(self):
        return self._n_inputs

    @property
    def n_outputs(self):
        return self._n_outputs
    
    @property
    def grid_variables(self):
        return self.left.grid_variables   
    
 
    
    def map_operator(self, op):
        """
        Mappa l'operatore dato a una funzione corrispondente.

        Args:
            op (str): Operatore come stringa.

        Returns:
            function: Funzione corrispondente all'operatore.

        """
        if op == "+":
            val = operator.add
        elif op == "/":
            val = operator.truediv
        elif op == "*":
            val = operator.mul
        elif op == "-":
            val = operator.sub
        elif op == '**':
            val == operator.pow
        else:
            val = None
        return val
    
    def _update_n_dim(self):
        """
        Controlla gli inputs e gli outputs dei sottomodelli per essere sicuro
        che le operazioni binarie siano supportate.
        corregge il numero di inputs/outputs del modello composito di conseguenza
        """

        if self.op_str in self.LINEAR_OPERATIONS:

            if self.left.n_dim != self.right.n_dim:
                raise ValueError("Number of dimensions do not match!")

            n_dim = self.left.n_dim
            n_inputs = self.left.n_inputs + self.right.n_inputs
            n_outputs = self.left.n_outputs

        elif self.op_str == self.COMPOSITE_OPERATION:
            if self.left.n_outputs != self.right.n_inputs:
                raise ValueError(
                    "Number of output for left must be equal to n_inputs of right!"
                )
            #if self.left.n_dim != self.right.n_dim:
            #    raise ValueError("Number of dimensions do not match!")

            n_dim = self.left.n_dim
            n_inputs = self.left.n_inputs 
            n_outputs = self.right.n_outputs
        
        return  n_dim, n_inputs, n_outputs
        
    def _collect_submodels(self):
        
        param_map = OrderedDict()
        submodels = []
        
        
        def dfs(node):
            nonlocal  param_map, submodels
            if not node:
                return
            
            if not node.left and not node.right:
                submodels.append(node.name)
                
                return           
            
            dfs(node.left)            
            dfs(node.right)
        
        dfs(self)

        return submodels
    
    def composite_structure(self):
        """
        Restituisce una stringa che rappresenta la logica con cui i sottomodelli sono uniti.

        Returns:
            str: Una stringa che rappresenta la struttura dell'albero binario del modello composito.
        """

        def helper(m, id_counter):
            if isinstance(m, CompositeModel):
                left_str, id_counter = helper(m.left, id_counter)
                right_str, id_counter = helper(m.right, id_counter)
                return f"({left_str} {m.op_str} {right_str})", id_counter
            else:
                return f"{m.name} [{id_counter}]", id_counter + 1

        structure, _ = helper(self, 0)
        return structure
    
    def _init_parameters(self):
        '''Crea un Nuovo ParameterHandler con i nomi cambiati dei parametri ma mappati
        agli stessi parametri originali
        '''

        parameters = ParameterHandler()
        n = 0

        def dfs(node):
            nonlocal n, parameters
            if node is None:
                return
            
            if not node.left and not node.right:
                for param in node:
                    name = param.name + f'_{n}'
                    
                    parameters.add_parameter(param, name=name)
                n += 1
                
            
            dfs(node.left)
            dfs(node.right)
        
        dfs(self)
        
        return parameters
    
    def __str__(self):
        """
        Restituisce una stringa che rappresenta il modello composito e i suoi parametri.

        Returns:
            str: Una stringa che rappresenta il modello composito, i modelli contenuti e i parametri liberi.
        """
        total_string = f"COMPOSITE MODEL NAME: {self.name} \n"
        total_string += (
            f"CONTAINED MODELS: {self.submodels}\n"
        )
        total_string += (
            f"GRID VARIABLES: {self.grid_variables}\n"
        )
        total_string += f"LOGIC: {self.composite_structure()}\n"
        total_string += f"FREE PARAMS: {self.n_free_parameters}\n"
        total_string += "-" * 60 + "\n"
        total_string += (
            f"{'':<4} {'NAME':<15} {'VALUE':<10} {'IS-FROZEN':<10} {'BOUNDS':<20} \n"
        )
        total_string += "-" * 60 + "\n"
        for i, (param_name, param) in enumerate(self.parameters.items()):
            value_str = f"{param.value:.2f}"
            bounds_str = f"({param.bounds[0]:.2f}, {param.bounds[1]:.2f})"
            frz = 'Yes' if param.frozen else 'No'
            total_string += f"{i:<4} {param_name:<15} {value_str:<10} {frz:<10} {bounds_str:<20}\n"
        return total_string



    def evaluate(self, *args, **kwargs):
        
        tmp = [0] * len(self.grid_variables)
        tmp.extend(self.parameters_values)

        if args:
            # grid, values
            for i in range(len(args)):
                tmp[i] = args[i]

        if kwargs:
            for i,key in enumerate(self.grid_variables):
                if key in kwargs:
                    tmp[i] = kwargs.pop(key)
                
            for key, val in kwargs.items():
                idx = self._map_name_to_index(key)
                tmp[idx + len(self.grid_variables)] = val

        grid = tmp[:len(self.grid_variables)]
        vals = tmp[len(self.grid_variables):]
        
        if  self.op_str in self.LINEAR_OPERATIONS:
            return self._operator(
                self.left.evaluate(*grid, *vals[: self.left.n_inputs]),
                self.right.evaluate(*grid, *vals[self.left.n_inputs:]),
            )
        elif self.op_str in self.COMPOSITE_OPERATION:
            left_res = self.left.evaluate(*grid, *vals[: self.left.n_inputs])
            return self.right.evaluate(*left_res)



    def __call__(self, **kwargs):
        # self.op(left(**args_left), right(**args_right))
        grid = {}
        for key in self.grid_variables:
            
            grid[key] = kwargs.pop(key)

        tmp = self.parameters_values_dict   # key modello comp e non submmodel

        if kwargs:
            for key in kwargs:
                if key not in self.grid_variables and self[key].frozen:
                    warnings.warn(
                        f"Parameter {key} is frozen, new value will be ignored"
                    )
                    continue
                else:
                    tmp[key] = kwargs[key]

        
        left_vals = {key:val for key,val in zip(self.left.parameters_keys, tmp.values()) if not self.left[key].frozen}
        right_vals = {key:val for key,val in zip(self.right.parameters_keys, list(tmp.values())[self.left.n_inputs:])if not self.right[key].frozen}
        
        if self.op_str in self.LINEAR_OPERATIONS:
            return self._operator(self.left(**grid, **left_vals), self.right(**grid, **right_vals))
        elif self.op_str in self.COMPOSITE_OPERATION:
            left_res = self.left(**grid, **left_vals)
            return self.right(*left_res)
        
    
    



        


In [10]:
print(model1.n_dim, model1.n_outputs, model1.n_inputs)
print(model2.n_dim, model2.n_outputs, model2.n_inputs)
print(model2.parameters_values_dict)
print(model2.n_dim)


1 1 2
1 1 2
{'mu': 1, 'sigma': 1}
1


In [11]:
Cmodel = model1 + model2 # no copy
#Cmodel.freeze_parameters()
Cmodel.set_parameters_values(mu_1 = 16.43)


print(Cmodel._map_name_to_index('sigma_1'))
print(Cmodel.evaluate(22,3,1,4,5))
print(Cmodel(x = 22,mu_0 = 3,mu_1 =4, sigma_1 =5))
print('-'*10)
print(Cmodel)


3
0.0001223803860227544
0.0001223803860227544
----------
COMPOSITE MODEL NAME: CompositeModel 
CONTAINED MODELS: ['Model1', 'Model2']
GRID VARIABLES: ['x']
LOGIC: (Model1 [0] + Model2 [1])
FREE PARAMS: 3
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS               
------------------------------------------------------------
0    mu_0            1.00       No         (-inf, inf)         
1    sigma_0         1.00       Yes        (-inf, inf)         
2    mu_1            16.43      No         (-inf, inf)         
3    sigma_1         1.00       No         (-inf, inf)         



In [12]:
Nmodel = Cmodel.copy() + Cmodel.copy()
Nmodel.freeze_parameters()
print(Nmodel._map_name_to_index('sigma_3'))
print(Nmodel)

print(Nmodel(x = 1))

7
COMPOSITE MODEL NAME: CompositeModel 
CONTAINED MODELS: ['Model1', 'Model2', 'Model1', 'Model2']
GRID VARIABLES: ['x']
LOGIC: ((Model1 [0] + Model2 [1]) + (Model1 [2] + Model2 [3]))
FREE PARAMS: 0
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS               
------------------------------------------------------------
0    mu_0            1.00       Yes        (-inf, inf)         
1    sigma_0         1.00       Yes        (-inf, inf)         
2    mu_1            16.43      Yes        (-inf, inf)         
3    sigma_1         1.00       Yes        (-inf, inf)         
4    mu_2            1.00       Yes        (-inf, inf)         
5    sigma_2         1.00       Yes        (-inf, inf)         
6    mu_3            16.43      Yes        (-inf, inf)         
7    sigma_3         1.00       Yes        (-inf, inf)         

0.7978845608028654


## TEST-CASE MODEL

In [17]:
def linear_model(x, a=1.0, b=0.0):
    return a * x + b

# Creiamo un modello lineare utilizzando la funzione lineare
model = Model.from_callable(
    func=linear_model,
    ndim=1,
    ninputs=3,
    noutputs=1,
    default_value=1.0,
    name="LinearModel",
)

print("Test 1: Verifica dei Parametri del Modello")
for param_name in model.parameters_keys:
    param = model[param_name]
    print(
        f"Nome: {param.name}, Valore: {param.value}, Limiti: {param.bounds}, Congelato: {param.frozen}"
    )
    
x_value = 2.0
y = model(x=x_value)
print(f"\nTest 2: Valutazione del modello a x={x_value}: y = {y}")


# Impostiamo a = 3.0
model.set_parameters_values(a=3.0)

# Valutiamo il modello
y = model(x=x_value)
print(
    f"\nTest 3: Dopo aver impostato a=3.0, valutazione del modello a x={x_value}: y = {y}"
)

# Congeliamo il parametro 'a'
model.freeze_parameters("a")

# Tentiamo di modificare 'a' durante la valutazione

with warnings.catch_warnings(record=True) as w:
    y = model(x=x_value, a=5.0)
    if w:
        print(f"\nTest 4: Avviso catturato: {w[-1].message}")

print(
    f"Dopo aver congelato a e tentato di impostare a=5.0, valutazione del modello a x={x_value}: y = {y}"
)

# Scongeliamo il parametro 'a'
model.unfreeze_parameters("a")

# Modifichiamo 'a' durante la valutazione
y = model(x=x_value, a=5.0)
print(
    f"\nTest 5: Dopo aver scongelato a e impostato a=5.0, valutazione del modello a x={x_value}: y = {y}"
)

# Impostiamo i limiti per 'a' tra 1.0 e 4.0
model.set_parameters_bounds(a=(1.0, 4.0))

# Proviamo a impostare 'a' fuori dai limiti
model.set_parameters_values(a=1.5)

# Verifichiamo il valore di 'a'
a_value = model["a"].value
print(
    f"\nTest 6: Dopo aver impostato a=1.5 con limiti (1.0, 4.0), valore effettivo di a: {a_value}"
)

# Creiamo una copia del modello
model_copy = model.copy()

# Modifichiamo un parametro nella copia
model_copy.set_parameters_values(a=2.0)

# Valutiamo entrambi i modelli
y_original = model(x=x_value)
y_copy = model_copy(x=x_value)

print(f"\nTest 7: Valutazione del modello originale a x={x_value}: y = {y_original}")
print(f"Valutazione del modello copiato a x={x_value} con a=2.0: y = {y_copy}")

print("\nTest 8: Rappresentazione Testuale del Modello")
model.name = 'Linear_Model_0'


print("\n test 9, accesso ai parametri, da congelati")
model.freeze_parameters('a')
model['a'].bounds = (-10,10)


model['a'].value = 2.22
print(f'nuovo valore deve essere 2.22 ed è {model["a"].value}')

print('\n test 10, iterazione sul modello')
for p in model:
    print(f'{p.name}, {p.value}, {p.bounds}')
print(model)


Test 1: Verifica dei Parametri del Modello
Nome: a, Valore: 1.0, Limiti: (-inf, inf), Congelato: True
Nome: b, Valore: 0.0, Limiti: (-inf, inf), Congelato: True

Test 2: Valutazione del modello a x=2.0: y = 2.0

Test 3: Dopo aver impostato a=3.0, valutazione del modello a x=2.0: y = 2.0

Test 4: Avviso catturato: Parameter a is frozen, new value will be ignored
Dopo aver congelato a e tentato di impostare a=5.0, valutazione del modello a x=2.0: y = 2.0

Test 5: Dopo aver scongelato a e impostato a=5.0, valutazione del modello a x=2.0: y = 10.0

Test 6: Dopo aver impostato a=1.5 con limiti (1.0, 4.0), valore effettivo di a: 1.5

Test 7: Valutazione del modello originale a x=2.0: y = 3.0
Valutazione del modello copiato a x=2.0 con a=2.0: y = 4.0

Test 8: Rappresentazione Testuale del Modello

 test 9, accesso ai parametri, da congelati
nuovo valore deve essere 2.22 ed è 1.5

 test 10, iterazione sul modello
a, 1.5, (1.0, 4.0)
b, 0.0, (-inf, inf)
MODEL NAME: Linear_Model_0 
FREE PARAMS: 0

/home/matteo/CosmoSynth/CosmoSynth/models/parameters.py:147: UserWarning: Parameter a is frozen, new bounds will be ignored!
  warnings.warn(f'Parameter {self.name} is frozen, new bounds will be ignored!')
/home/matteo/CosmoSynth/CosmoSynth/models/parameters.py:119: UserWarning: Parameter a is frozen, new value will be ignored!
  warnings.warn(f'Parameter {self.name} is frozen, new value will be ignored!')


## TEST CASE COMPOSITE MODEL

In [18]:
def quadratic_model(x, c, d):
    return c * x**2 + d

# Creazione del modello lineare
linear = Model.from_callable(
    func=linear_model,
    ndim=1,
    ninputs=2,
    noutputs=1,
    default_value=1.0,
    name="LinearModel",
)

# Creazione del modello quadratico
quadratic = Model.from_callable(
    func=quadratic_model,
    ndim=1,
    ninputs=2,
    noutputs=1,
    default_value=1.0,
    name="QuadraticModel",
)

composite = linear + quadratic

print(composite)
print("Test 1: Verifica dei Parametri del Modello Composito")
for param_name in composite.parameters_keys:
    param = composite[param_name]
    print(
        f"Nome: {param.name}, Valore: {param.value}, Limiti: {param.bounds}, Congelato: {param.frozen}"
    )

x_value = 2.0
y = composite(x=x_value)
print(f'\nvalutazione modello lineare a {x_value}: linear = {linear(x = 2)}')
print(f"\nvalutazione modello quadratico a {x_value}: quadratic = {quadratic(x = 2)}")
print(f"\nTest 2: Valutazione del modello composito a x={x_value}: y = {y}")

# Congeliamo il parametro 'c'
composite.freeze_parameters("c_1")

# Tentiamo di modificare 'c' durante la valutazione
with warnings.catch_warnings(record=True) as w:
    y = composite(x=x_value, c_1=5.0)
    if w:
        print(f"\nTest 4: Avviso catturato: {w[-1].message}")

print(
    f"Dopo aver congelato c e tentato di impostare c=5.0, valutazione del modello composito a x={x_value}: y = {y}"
)

# Scongeliamo il parametro 'c'
composite.unfreeze_parameters("c_1")

# Modifichiamo 'c' durante la valutazione
y = composite(x=x_value, c_1=5.0)
print(
    f"\nTest 5: Dopo aver scongelato c e impostato c=5.0, valutazione del modello composito a x={x_value}: y = {y}"
)

#composite.unfreeze_parameters()

# Impostiamo i limiti per 'a' tra 1.0 e 4.0
#composite.set_parameters_bounds(a_0=(1.0, 4.0))
composite['a_0'].bounds = (-10,10)
composite.set_parameters_bounds(a_0 = (-2,2))
# Proviamo a impostare 'a' fuori dai limiti
try:
    composite.set_parameters_values(a_0=5.0)
except ValueError as e:
    print(f"\nTest 6: Errore catturato: {e}")

# Verifichiamo il valore di 'a'
a_value = composite["a_0"].value
print(
    f"Dopo aver tentato di impostare a=5.0 con limiti (1.0, 4.0), valore effettivo di a: {a_value}"
)

print(composite)

COMPOSITE MODEL NAME: CompositeModel 
CONTAINED MODELS: ['LinearModel', 'QuadraticModel']
GRID VARIABLES: ['x']
LOGIC: (LinearModel [0] + QuadraticModel [1])
FREE PARAMS: 2
------------------------------------------------------------
     NAME            VALUE      IS-FROZEN  BOUNDS               
------------------------------------------------------------
0    a_0             1.00       Yes        (-inf, inf)         
1    b_0             0.00       Yes        (-inf, inf)         
2    c_1             1.00       No         (-inf, inf)         
3    d_1             1.00       No         (-inf, inf)         

Test 1: Verifica dei Parametri del Modello Composito
Nome: a, Valore: 1.0, Limiti: (-inf, inf), Congelato: True
Nome: b, Valore: 0.0, Limiti: (-inf, inf), Congelato: True
Nome: c, Valore: 1.0, Limiti: (-inf, inf), Congelato: False
Nome: d, Valore: 1.0, Limiti: (-inf, inf), Congelato: False

valutazione modello lineare a 2.0: linear = 2.0

valutazione modello quadratico a 2.0: quad

/home/matteo/CosmoSynth/CosmoSynth/models/parameters.py:147: UserWarning: Parameter a is frozen, new bounds will be ignored!
  warnings.warn(f'Parameter {self.name} is frozen, new bounds will be ignored!')


In [15]:
'''def func(a, b, c):
    return  a+b+c


model = Model.from_callable(func, name='Test')
model.freeze_parameters("b")
print(model)
print(model._binary_freeze_map)
print(model.n_dim, model.n_inputs, model.n_outputs)
model(1,4)  # '2' is intended for 'c' but gets assigned to 'c' due to 'b' being frozen

model2 = model.copy()
## test per vedere se  arrivo allo stesso parametro
## confermo copia modello crea nuovi parametri
print(model.parameters._parameters)
print(model2.parameters._parameters)
'''

'def func(a, b, c):\n    return  a+b+c\n\n\nmodel = Model.from_callable(func, name=\'Test\')\nmodel.freeze_parameters("b")\nprint(model)\nprint(model._binary_freeze_map)\nprint(model.n_dim, model.n_inputs, model.n_outputs)\nmodel(1,4)  # \'2\' is intended for \'c\' but gets assigned to \'c\' due to \'b\' being frozen\n\nmodel2 = model.copy()\n## test per vedere se  arrivo allo stesso parametro\n## confermo copia modello crea nuovi parametri\nprint(model.parameters._parameters)\nprint(model2.parameters._parameters)\n'

In [16]:
'''def simple_test(a,b,c):
    return a +b +c

test = Model.from_callable(simple_test)

test.freeze_parameters(b = 2)
test.set_parameters_values(a = 10)

print(test)
print(test())
print(test.evaluate(1,1, 1))'''

'def simple_test(a,b,c):\n    return a +b +c\n\ntest = Model.from_callable(simple_test)\n\ntest.freeze_parameters(b = 2)\ntest.set_parameters_values(a = 10)\n\nprint(test)\nprint(test())\nprint(test.evaluate(1,1, 1))'